# Basic CNN Training (PyTorch)

This notebook trains a CNN classifier on images in `spatial-tech/data_collection/data/clean_train_data` for 4 classes: `click`, `mouse`, `palm`, and `thumb`.

In [4]:
from pathlib import Path
import random

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

# Reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

# Dataset path: clean_train_data
candidate_paths = [
    Path("spatial-tech/data_collection/data/clean_train_data"),
    Path("../../data_collection/data/clean_train_data"),
    Path("../data_collection/data/clean_train_data"),
    Path("data_collection/data/clean_train_data"),
]

data_dir = None
for p in candidate_paths:
    if p.exists() and p.is_dir():
        data_dir = p.resolve()
        break

if data_dir is None:
    raise FileNotFoundError(
        "Could not find dataset folder. Expected data at spatial-tech/data_collection/data/clean_train_data."
    )

print(f"Dataset directory: {data_dir}")

# Output path under basic_cnn (used by the separate save cell)
output_candidate_paths = [
    Path("spatial-tech/models/basic_cnn"),
    Path("."),
]

output_dir = None
for p in output_candidate_paths:
    # Prefer the explicit project path when running from repo root,
    # otherwise fall back to current directory (notebook folder).
    if p.exists() and p.is_dir():
        output_dir = p.resolve()
        break

if output_dir is None:
    output_dir = Path(".").resolve()

model_path = output_dir / "basic_cnn.pth"
meta_path = output_dir / "basic_cnn_meta.pth"
print(f"Model save path: {model_path}")

Using device: mps
Dataset directory: /Users/aryamanwade/Desktop/ml_projects/desktop_cnn/spatial-tech/data_collection/data/clean_train_data
Model save path: /Users/aryamanwade/Desktop/ml_projects/desktop_cnn/spatial-tech/models/basic_cnn/spatial-tech/models/basic_cnn/basic_cnn_clean_train_data.pth


In [3]:
# Hyperparameters
IMG_SIZE = 128
BATCH_SIZE = 32
VAL_SPLIT = 0.15
EPOCHS = 30
LEARNING_RATE = 1e-3
EARLY_STOPPING_PATIENCE = 4
EXPECTED_CLASSES = ["click", "mouse", "palm", "thumb"]

# Data is already 128x128, so no resize transform is applied.
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

full_dataset = datasets.ImageFolder(root=str(data_dir), transform=transform)
class_names = full_dataset.classes
num_classes = len(class_names)

if sorted(class_names) != sorted(EXPECTED_CLASSES):
    raise ValueError(
        f"Unexpected classes found: {class_names}. Expected exactly: {EXPECTED_CLASSES}"
    )

if num_classes < 2:
    raise ValueError("Need at least 2 classes to train a classifier.")

# Stratified split: remove 15% of each class into validation set.
rng = random.Random(SEED)
class_to_indices = {i: [] for i in range(num_classes)}
for idx, target in enumerate(full_dataset.targets):
    class_to_indices[target].append(idx)

train_indices = []
val_indices = []
for class_idx, indices in class_to_indices.items():
    rng.shuffle(indices)
    n_val = max(1, int(len(indices) * VAL_SPLIT))
    if n_val >= len(indices):
        raise ValueError(
            f"Class '{class_names[class_idx]}' has too few samples for a 15% val split."
        )

    val_indices.extend(indices[:n_val])
    train_indices.extend(indices[n_val:])

train_dataset = Subset(full_dataset, train_indices)
val_dataset = Subset(full_dataset, val_indices)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Total images: {len(full_dataset)}")
print(f"Train images: {len(train_dataset)}")
print(f"Val images: {len(val_dataset)}")
print(f"Classes: {class_names}")

for class_idx, class_name in enumerate(class_names):
    total_in_class = len(class_to_indices[class_idx])
    val_in_class = max(1, int(total_in_class * VAL_SPLIT))
    train_in_class = total_in_class - val_in_class
    print(f"  - {class_name}: train={train_in_class}, val={val_in_class}")


class BasicCNN(nn.Module):
    def __init__(self, n_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * (IMG_SIZE // 8) * (IMG_SIZE // 8), 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


model = BasicCNN(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)


def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / total
    acc = correct / total
    return avg_loss, acc


best_val_loss = float("inf")
epochs_without_improvement = 0
best_state_dict = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        running_total += labels.size(0)

    train_loss = running_loss / running_total
    train_acc = running_correct / running_total

    val_loss, val_acc = evaluate(model, val_loader)

    print(
        f"Epoch [{epoch}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f} Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} Val Acc: {val_acc:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0
        best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print(
                f"Early stopping triggered: validation loss did not improve for "
                f"{EARLY_STOPPING_PATIENCE} epochs."
            )
            break

if best_state_dict is not None:
    model.load_state_dict(best_state_dict)
    model.to(device)
    print(f"Loaded best model weights with val loss: {best_val_loss:.4f}")

Total images: 9123
Train images: 7299
Val images: 1824
Classes: ['click', 'mouse', 'palm', 'thumb']
Epoch [1/10] Train Loss: 0.3131 Train Acc: 0.8811 | Val Loss: 0.0469 Val Acc: 0.9879


KeyboardInterrupt: 

In [ ]:
# Final evaluation on clean_test_data (run this before saving)
test_candidate_paths = [
    Path("spatial-tech/data_collection/data/clean_test_data"),
    Path("../../data_collection/data/clean_test_data"),
    Path("../data_collection/data/clean_test_data"),
    Path("data_collection/data/clean_test_data"),
]

test_dir = None
for p in test_candidate_paths:
    if p.exists() and p.is_dir():
        test_dir = p.resolve()
        break

if test_dir is None:
    raise FileNotFoundError(
        "Could not find test dataset folder. Expected data at spatial-tech/data_collection/data/clean_test_data."
    )

test_dataset = datasets.ImageFolder(root=str(test_dir), transform=transform)
if sorted(test_dataset.classes) != sorted(EXPECTED_CLASSES):
    raise ValueError(
        f"Unexpected test classes found: {test_dataset.classes}. Expected exactly: {EXPECTED_CLASSES}"
    )

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

test_loss, test_acc = evaluate(model, test_loader)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Acc:  {test_acc:.4f}")

# Per-class accuracy
class_correct = {name: 0 for name in class_names}
class_total = {name: 0 for name in class_names}

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1)

        for label, pred in zip(labels, preds):
            class_name = class_names[label.item()]
            class_total[class_name] += 1
            if pred.item() == label.item():
                class_correct[class_name] += 1

print("Per-class accuracy:")
for class_name in class_names:
    total = class_total[class_name]
    acc = class_correct[class_name] / total if total > 0 else 0.0
    print(f"  - {class_name}: {acc:.4f} ({class_correct[class_name]}/{total})")

In [ ]:
# Save model and metadata (run this cell only when you want to save)
torch.save(model.state_dict(), model_path)

torch.save(
    {
        "class_names": class_names,
        "img_size": IMG_SIZE,
        "num_classes": num_classes,
        "model_name": "BasicCNN",
    },
    meta_path,
)

print(f"Saved model weights to: {model_path.resolve()}")
print(f"Saved metadata to: {meta_path.resolve()}")